# U01 Python、資料處理與資料管理入門

以成績資料練習 Python、檔案處理、NumPy、pandas 與 matplotlib。
從存檔、資料驗證與副本覆寫的結果，辨認資料管理需求。
所有練習使用合成資料。

## 作答方式

依序執行程式格。有 TODO 的格子先自行作答，再查閱折疊解答。
保持題目的輸入與預期結果不變，修改作答區；參考解答也填回該題的作答區。
「待核對」要求自行比較結果，「未通過」表示答案不符；執行完成不等於作答正確。
既有案例只檢查部分輸入，仍須依題目的資料規則補測其他情況。

# 課程與專題

- 評分：專題 70%、平時 30%。
- 每人獨立完成教師指派題目；需求與驗收方式以專題規範為準。
- 本單元練習用 Python 表達資料規則、整理檔案、分析與繪圖，並辨認共用資料的管理需求。
- 報告當天使用自己的 Colab 展示，結束後繳交 notebook。
- 以講義為主，Python 補充讀物使用 Practical Python Programming。

## AI 協作規則

1. 可以使用 AI 協助寫程式、解讀錯誤與提出測試案例。
2. 記錄使用工具、主要問題，以及如何核對或修正回答。
3. 對交付內容負責，必須能解釋程式與驗證結果。

## Colab 的執行與儲存

| 動作 | 操作 |
|---|---|
| 執行目前格並移到下一格 | Shift+Enter |
| 原地執行 | Ctrl+Enter |
| 重新執行完整程式 | 執行階段 → 全部執行 |
| 中斷執行 | 執行階段 → 中斷執行 |
| 保存自己的 notebook | 檔案 → 在雲端硬碟中儲存副本 |

執行階段中的變數與暫存檔可能消失；保存 notebook 不等於保存程式產生的資料檔。
重要資料檔須另外下載或保存至雲端硬碟。
本講義使用獨立練習目錄，不覆蓋既有資料。更換執行階段後，從第一格重新執行。

# 為什麼需要資料庫

你是系上助教，用 CSV 管理 800 位學生、40 門課的成績。
兩位助教各更正一名學生的成績，也都存檔了。兩筆更正會全部留下嗎？

以下用少量合成資料重現操作，每次完整執行建立新的練習目錄。

In [ ]:
import tempfile
from pathlib import Path
import pandas as pd

opening_dir = Path(tempfile.mkdtemp(prefix="u01-opening-"))
print("資料目錄：", opening_dir)

## 兩位助教各改一筆，兩筆修改都會留下嗎？

A、B 各自開啟同一份成績檔。A 把 S001 的 80 改成 85，先存檔；B 把 S002 的 65 改成 70，再存檔。
程式固定依「A 讀、B 讀、A 寫、B 寫」執行，沒有啟動兩個同時運行的程式。

In [ ]:
opening_grades_path = opening_dir / "grades.csv"
pd.DataFrame({"sid": ["S001", "S002"], "grade": [80, 65]}).to_csv(
    opening_grades_path, index=False)
opening_a = pd.read_csv(opening_grades_path)  # A 取得一份副本。
opening_b = pd.read_csv(opening_grades_path)  # B 也取得原始副本。
opening_a.loc[opening_a["sid"] == "S001", "grade"] = 85
opening_a.to_csv(opening_grades_path, index=False)
print("A 存檔後：\n" + pd.read_csv(opening_grades_path).to_string(index=False))
opening_b.loc[opening_b["sid"] == "S002", "grade"] = 70
opening_b.to_csv(opening_grades_path, index=False)
opening_saved = pd.read_csv(opening_grades_path)
print("B 存檔後：\n" + opening_saved.to_string(index=False))
assert opening_saved["grade"].tolist() == [80, 70]
assert opening_saved["sid"].is_unique
assert opening_saved["grade"].between(0, 100).all()
print("學號沒有重複、成績都在範圍內，但 A 保存的 85 分消失了。")

## 為什麼改不同學生，仍會覆蓋？

A、B 讀出的是記憶體中的獨立副本。修改副本不會自動改變檔案或另一份副本。
to_csv 寫回的是整份表格；B 保存時，手上的 S001 仍是 80，因此覆蓋了 A 的 85。

最後的資料仍符合學號不重複、成績介於 0–100 的規則；資料驗證不能代替修改協調。
只在寫入前重讀也不保證安全，另一人仍可能在讀取與寫入之間保存修改。

## 78 分只寫到 7，讀檔會報錯嗎？

預定保存三筆成績：S001 的 85 分、S002 的 78 分、S003 的 92 分。
如果只寫到 S002 的第一個數字，讀檔程式會發現資料不完整嗎？
本例刻意建立截短的檔案，不是實際當機或斷電測試。

In [ ]:
opening_expected = pd.DataFrame({
    "sid": ["S001", "S002", "S003"], "grade": [85, 78, 92]})
opening_partial_path = opening_dir / "all_grades.csv"
with open(opening_partial_path, "w", encoding="utf-8") as file:
    file.write("sid,grade\n")
    file.write("S001,85\n")
    file.write("S002,7")  # 78 只寫出 7，第三筆尚未寫入。
opening_partial = pd.read_csv(opening_partial_path)
print("預定保存：\n" + opening_expected.to_string(index=False))
print("實際讀到：\n" + opening_partial.to_string(index=False))
assert opening_partial["grade"].tolist() == [85, 7]
assert "S003" not in opening_partial["sid"].tolist()
assert opening_partial["grade"].between(0, 100).all()
print("讀檔成功，成績也在範圍內，但 78 變成 7，S003 不見了。")

## 能讀取，不代表保存完整

對 CSV 解析器而言，7 是合法數字；它不知道原本要保存 78，也不知道還有第三筆。
因此「讀檔成功」與「成績介於 0–100」都無法證明這次保存完整。
必須核對預定保存的資料，並安排寫入中斷時如何保留或還原完整版本。

## 存檔不等於已經備份

備份是另外保存可還原的副本。只留下會被反覆覆寫的同一檔案，不能還原覆寫前的內容。
備份應另存位置，避免修改原檔時連備份一起覆蓋；是否可用，須實際讀回並核對內容。

## 同一位學生，檔案裡卻有三種學號

同一門課的三份成績名冊使用 S001、s001、S0 01。依正式名冊確認，這三筆屬於同一位學生。
姓名還多了一個空白，成績有數字，也有「八十」。第四筆缺少學號，成績是 999。
把這些資料存成 CSV 再讀回，會自動被拒絕或整理成同一人嗎？

In [ ]:
opening_bad_rows = [
    {"sid": "S001", "name": "林佳蓉", "grade": 88},
    {"sid": "s001", "name": "林佳蓉", "grade": 90},
    {"sid": "S0 01", "name": "林 佳蓉", "grade": "八十"},
    {"sid": None, "name": "路人甲", "grade": 999},
]
opening_bad_path = opening_dir / "mixed_roster.csv"
pd.DataFrame(opening_bad_rows).to_csv(opening_bad_path, index=False)
opening_bad = pd.read_csv(opening_bad_path)
print("存檔、讀檔都成功：\n" + opening_bad.to_string(index=False))
assert opening_bad["sid"].iloc[:3].tolist() == ["S001", "s001", "S0 01"]
assert pd.isna(opening_bad.loc[3, "sid"])
assert opening_bad["grade"].tolist() == ["88", "90", "八十", "999"]
print("學號原文有", opening_bad["sid"].nunique(), "個非空不同值；前三筆其實屬於同一人。")

## 資料規則不能只靠某一個輸入畫面

資料完整性是資料符合已訂定的規則。本例需要統一學號表示、禁止缺少學號，並限定成績格式與範圍。
哪些紀錄屬於同一人須以正式名冊核對；不能只憑姓名相似就合併。

CSV 不會自行執行這些規則。即使一個輸入表單有驗證，另一個程式仍能直接寫入這些內容。
要保護共用資料，就必須讓所有寫入途徑遵守相同規則。

## 資料庫與資料庫管理系統

資料庫是依用途組織的資料集合，例如學生名冊、成績與課程資料。
資料庫管理系統（DBMS）是管理這些資料的軟體，負責儲存、讀取及控制修改。
資料是被管理的內容；系統是接受操作並執行規則的軟體，兩者不同。

應用程式是使用者操作的介面及流程，例如教師輸入成績的程式。
多個應用程式可以交由同一個 DBMS 管理資料，不必各自直接覆寫共用檔案。

## 統一管理哪些事情？

合法的舊副本仍可能覆蓋別人的修改；能讀取的檔案仍可能缺少內容；存檔成功也不代表符合資料規則。
DBMS 將資料保存、規則檢查與不同操作的協調放在同一個管理系統中。

DBMS 提供管理機制，但不會自動知道合理成績或正確的修改意圖。
開發者仍須設定規則、選擇修改方式並處理拒絕結果；使用 DBMS 不表示不必測試或備份。

## 選系統前，先描述需求

同樣保存成績，個人離線登記、教師共同修改、全校歷年統計，需要處理的問題不同。
正式資料是業務目前採用並持續維護的紀錄，不是每個人手上的任意副本。

選型先回答：

- 誰讀取、誰修改？是否有多人或多個程式同時修改？
- 每次處理少數紀錄，還是讀取大量資料做統計？
- 是否必須離線使用？結果可以比最新資料晚多久？
- 哪些修改不能遺失？誰負責備份、還原及系統維護？

資料筆數只回答其中一部分，不能單憑「資料很多」決定產品。

## 登記一筆訂單，與統計整年訂單

工作負載是系統實際承受的讀取、修改及同時使用情況。

| 工作 | 常見需求 | 名稱 |
|---|---|---|
| 查詢一張訂單、售出商品並扣減庫存 | 經常處理少數紀錄，必須協調同時修改 | 日常交易處理（OLTP） |
| 按月份統計全年銷售、比較各店營收 | 讀取大量紀錄，計算分組結果 | 分析處理（OLAP） |

這裡的交易指訂購、付款等業務操作。同一個應用可以同時有這兩類工作；名稱不是產品的互斥分類。

## 關聯式資料庫：以表格組織資料

一張表的每列代表一筆紀錄，每欄代表一項屬性。
例如學生表保存學號與姓名，成績表保存學號、課程與成績；透過學號對照同一位學生。
以這類表格及其關聯組織資料的系統，稱為關聯式資料庫系統。

SQLite、PostgreSQL、MySQL 都屬於這類；它們的部署方式與使用限制並不相同。
學生、訂單、庫存等資料常能用表格表達，但表格本身不會自動決定資料規則。

## 嵌入式與伺服器式

資料庫引擎是實際執行資料儲存、讀取與修改的程式。
程序是正在執行的程式。伺服器是提供服務的程式，不一定在另一台電腦。

| 部署方式 | 程式如何使用資料庫 | 例子 |
|---|---|---|
| 嵌入式 | 應用程式載入資料庫引擎，在自己的程序內操作資料 | SQLite |
| 客戶端／伺服器式 | 應用程式作為客戶端，向獨立運作的資料庫服務提出要求 | PostgreSQL、MySQL |

關閉一個客戶端，不代表資料庫服務也停止；嵌入式引擎通常隨應用程序結束。
引擎停止與資料消失是兩件事：已正確保存的資料仍可留在磁碟。

## SQLite：應用程式管理本機資料

[SQLite](https://www.sqlite.org/whentouse.html) 不要求另外啟動資料庫伺服器，適合離線記帳、裝置資料及單機收銀等需求。
它將資料保存於本機資料庫檔案，應用程式負責提供操作介面。

同一資料庫可供多個程式讀取，但同一時間只能有一個寫入者；其他寫入可能需要等待。
這不表示只能一人使用，而是要評估修改頻率及可以等待多久。
若多台電腦要共同修改，不宜只把檔案放進共用網路資料夾就當成資料庫服務。

## PostgreSQL：由服務集中管理共用資料

[PostgreSQL](https://www.postgresql.org/docs/current/tutorial-arch.html) 接受多個客戶端的要求。
例如售票網站、櫃檯程式與管理介面，可以共同使用資料庫服務，而不是交換各自的檔案副本。
MySQL 也是常見的伺服器型關聯式系統。

多個應用共同讀寫、需要限制不同使用者可做的操作時，可評估這類部署。
同時也需要有人管理服務、決定誰能使用、安裝更新，以及處理備份與故障。
換成伺服器型產品，不表示同一張票就不會被重複賣出；修改流程仍須正確設計。

## 帳號與權限回答不同問題

帳號是系統用來識別使用者或應用程式的身分名稱。
身分確認是判定「你是誰」；權限是規定「你可以讀取或修改哪些資料」。
例如教師可修改自己課程的成績，學生只能查看自己的成績，統計程式只需讀取資料。

隱藏畫面上的修改按鈕，不等於阻止修改；系統仍須檢查實際收到的操作要求。
共用同一個高權限帳號，也無法靠帳號分辨不同使用者。
選型時應確認系統如何限制操作，以及誰負責設定；有帳號功能不表示權限已設定正確。

## DuckDB：大量資料的彙整分析

[DuckDB](https://duckdb.org/why_duckdb) 主要面向分析工作，例如讀取大量歷年紀錄，按學期、系所計算人數與平均。
它可以嵌入分析程式，不必為本機分析另外管理一個資料庫服務。

選擇時應用實際資料量與報表需求測試執行時間、記憶體和磁碟用量。
大量統計適合評估 DuckDB，不表示它只能讀取資料，也不表示 PostgreSQL 不能做報表。
一個系統也可以用 PostgreSQL 接受日常修改，再將資料副本交給分析工具處理。

## 鍵值資料與快取

鍵值資料用一個名稱找出對應內容，例如以商品編號取得商品介紹。
快取是保存可再次取得或重新計算的資料副本，減少重複讀取或計算。
[Redis](https://redis.io/docs/latest/develop/get-started/data-store/) 是常見的鍵值資料系統，可用於這類工作。

例如把熱門商品介紹暫存起來，減少重複讀取來源資料；商品修改後，仍須處理快取何時更新或移除。
Redis 不只用於快取，也有資料保存機制；但不能因此假定任何設定都不會遺失資料。
若把它當快取，應先指定正式資料在哪裡，以及副本遺失或過期時如何重建。

## 文件資料：把相關內容放在一起

文件是一筆包含多項內容的紀錄，裡面還能包含另一筆紀錄或一組項目，不是指文書軟體的檔案。
例如一件商品同時包含名稱、規格與多個尺寸；不同類別的商品可能需要不同屬性。

[MongoDB](https://www.mongodb.com/docs/manual/data-modeling/) 以文件組織資料。
經常一起讀取的商品內容可放在同一份文件；選型仍須考慮內容如何修改，以及是否會重複保存同一項資料。
欄位可以不同，不等於不必設計資料格式或檢查規則；關聯式系統也能處理部分這類需求。

## 向量檢索：尋找意思相近的內容

關鍵字搜尋比對文字；若想找不同措辭但意思相近的判決書，可評估向量檢索。
向量是一串數值。文字模型把文章轉成向量，再依向量的接近程度找出候選文章。

可使用專門提供向量檢索的系統，或在 PostgreSQL 加裝 [pgvector](https://github.com/pgvector/pgvector)。
pgvector 是擴充套件，為既有系統增加向量儲存與檢索功能。
儲存與比對向量，不等於資料庫會自行產生文字向量；也不代表最接近的文章必定正確。
必須保留原文，並用已知相關文章的問題檢查搜尋品質。

## 選型練習：離線收銀或記帳

一台收銀機需要在斷網時繼續記錄訂單，店家要求已確認的訂單不能因斷電遺失。
提出候選系統、兩項理由，以及還要確認或測試的條件。再判斷個人離線記帳是否適用相同方向。

<details><summary>參考方向</summary>

可先評估 SQLite：資料在本機，不必另外管理資料庫服務，也不依賴網路才能操作。
但選了產品還不能證明斷電安全，須確認何時回報保存成功、儲存裝置及設定，並測試中斷與還原。
個人離線記帳也符合這種部署需求；若多台收銀機要共用庫存，必須另行處理共同修改與斷網後的資料整合。

</details>

## 選型練習：一億筆教學評量

學校要把一億筆歷年教學評量，按學期與系所彙整成儀表板。
提出候選系統與理由。只知道筆數，能決定部署方式嗎？還缺哪些需求？

<details><summary>參考方向</summary>

這是大量資料彙整，可先評估 DuckDB 等分析型系統，用實際報表測試。
還要確認資料多久更新、報表允許多舊、同時有多少人查閱，以及分析在哪台機器執行。
本機產生每日報表，與很多人查閱持續更新的結果，是不同需求；不能只靠一億筆就決定完整系統。

</details>

## 選型練習：多人搶最後一張票

網站與櫃檯共同使用票券庫存，尖峰每秒可能有一萬人提出購票要求。
售票程式需要修改庫存，統計程式只需讀取。
提出候選系統、選擇理由，以及選定產品後仍須驗證的事情。

<details><summary>參考方向</summary>

可評估 PostgreSQL 等伺服器型關聯式系統，由服務管理多個客戶端，並限制各帳號的操作。
仍須設計購票與庫存修改流程，測試多人競爭最後一張票、重複要求及操作失敗。
每秒一萬人是待測的需求，不是使用 PostgreSQL 就能保證的效能。

</details>

## 選型練習：找出意思相近的判決書

使用者輸入一段文字，希望找到意思最接近的十篇判決書，不要求用詞完全相同。
提出候選方案，區分資料儲存、文字轉換與結果驗證各自的工作。

<details><summary>參考方向</summary>

可評估向量檢索系統，或 PostgreSQL 搭配 pgvector。
文字模型產生向量，系統保存文章與向量並找出候選結果；最後用已知相關文章的測試問題檢查品質。
還要確認文章數量、更新頻率與可接受的錯誤結果比例。向量相近不等於法律論點或事實必定相同。

</details>

# 1 Python 基礎與練習

## 值、型別、變數、f-string

| 型別 | 長相 | 用在 |
|---|---|---|
| `int`／`float` | `42`、`3.14` | 數量、金額、成績 |
| `str` | `"林佳蓉"` | 文字（單雙引號皆可） |
| `bool` | `True`／`False` | 條件 |
| `NoneType` | `None` | 「沒有值」 |

**f-string**：`f"{value}"` 將值放進文字，`{value:.2f}` 顯示兩位小數。

`=` 指定值；`print()` 顯示結果，`type()` 顯示型別。`#` 到行尾是註解。
`==` 比較相等，`!=` 比較不相等，結果是 True 或 False；`is None`／`is not None` 檢查有無值。
`.1f` 顯示一位小數；`<8`／`>8` 靠左／靠右對齊，`,` 加千分位，`.1%` 顯示百分比。

In [ ]:
sid = "S001"
score = 87.456
n_course = 3

print(f"{sid} 修了 {n_course} 門課，平均 {score:.1f} 分")     # :.1f = 一位小數
print(f"{'學號':<8}{'平均':>8}")                              # <8 靠左、>8 靠右（對齊表格）
print(f"{sid:<8}{score:>8.1f}")
print(f"千分位：{1234567:,}　百分比：{0.949:.1%}")
print(type(sid), type(score), type(None))

In [ ]:
# 練習 1：用 f-string 組出「S001 的 3 科平均是 87.5 分」（一位小數）
msg = ""      # TODO

print('練習 1：待核對', msg, "；期望", "S001 的 3 科平均是 87.5 分")

## list 與切片：一串資料

```python
xs = [88, 55, 74, 92]     # 建立
xs[0]                     # 取第一個；索引從 0 起算
xs[-1]                    # 取最後一個
xs[1:3]                   # 切片：位置 1 到 2（「含頭不含尾」）
xs.append(60)             # 加一筆
len(xs)
sum(xs)
sorted(xs)
max(xs)
```
`len()` 算筆數、`sum()` 加總、`sorted()` 回傳排序副本、`max()` 取最大值。
`reverse=True` 改為由大到小排序；`/` 是除法。`[:2]` 從開頭取兩項。
`xs.append(60)` 是對 xs 呼叫 append 方法；點號後接物件提供的操作。

In [ ]:
grades = [88, 55, 74, 92, 61]
print("第一筆", grades[0], "／最後一筆", grades[-1])
print("中間三筆", grades[1:4])
print("由高到低", sorted(grades, reverse=True))
print("前二名", sorted(grades, reverse=True)[:2])
print(f"共 {len(grades)} 筆、總分 {sum(grades)}、平均 {sum(grades)/len(grades):.1f}")

In [ ]:
# 練習 2：取出 grades「去掉最高與最低之後」的清單（修剪平均的前置動作；提示：sorted + 切片）
trimmed = []      # TODO

print('練習 2：待核對', sorted(trimmed), "；期望", [61, 74, 88])

## dict、tuple、set：對照表、打包、集合

| 容器 | 長相 | 用途 | 修改特性 |
|---|---|---|---|
| `dict` | `{"S001": 88}` | 查詢表：鍵 → 值 | 同一鍵再次指定會覆寫 |
| `tuple` | `("S001", 88)` | 不可改的一列 | 不能就地改其中一項 |
| `set` | `{"統計", "資訊"}` | 不重複的值 | 不保證迭代順序 |

`d[key]` 取值，`d.get(key, default)` 在缺鍵時回傳預設值；省略 default 時回傳 None。
`d.keys()` 提供鍵，`d.items()` 提供 `(鍵, 值)` 配對。`list(容器)` 產生內容清單。
`dict(pairs)` 可把鍵值配對建成 dict；`sid, name, grade = row` 按位置拆開 tuple。
`in` 檢查成員，`A - B` 取集合差集；`set()` 建立空集合，`{}` 是空 dict。

In [ ]:
score_of = {"S001": 88, "S002": 55, "S003": 74}
print("S002 的成績：", score_of["S002"])
print("查無此人給預設：", score_of.get("S999", "（沒有這個人）"))
score_of["S004"] = 92                          # 新增／覆寫
print("所有學號：", list(score_of.keys()))

row = ("S001", "林佳蓉", 88)                    # tuple：一列資料
sid, name, g = row                              # 拆包（unpacking）
print(f"{sid} {name} {g} 分")

depts = {"統計", "資訊", "統計", "數學"}         # set 自動去重
print("不重複系所：", depts, "｜有統計嗎？", "統計" in depts)

## 流程控制：if／for／while

`if`、`elif`、`else` 依序選擇第一個成立的分支。冒號後的縮排表示區塊範圍。
`>=` 表示大於或等於；`<`、`<=`、`>` 分別表示小於、小於或等於、大於。
`and` 要求兩個條件都成立，`or` 表示任一條件成立，`not` 反轉條件。
分數由高門檻往下判斷，才能讓高分進入正確分類。

In [ ]:
# if / elif / else：成績 → 等第
g = 85
if g >= 90:
    label = "A"
elif g >= 80:
    label = "B"
elif g >= 60:
    label = "C"
else:
    label = "F"
print(g, "→", label)

## 逐筆配對與編號

`for x in 來源` 依序取值。`zip(names, grades)` 配對相同位置，長度不同時停在較短的一方。
`enumerate(..., start=1)` 提供編號；`i, (name, grade)` 把編號與配對拆開。
`a if 條件 else b` 依條件產生一個值。以下兩份清單長度相同。
`+=` 將值加進原變數；可以配合迴圈累積筆數與總分。

In [ ]:
# for + enumerate + zip：逐筆配對並加上編號
names = ["佳蓉", "威廷", "雅筑"]
grades = [88, 55, 74]
for i, (name, g) in enumerate(zip(names, grades), start=1):
    tag = "（及格）" if g >= 60 else "（不及格）"
    print(f"{i}. {name} {g} {tag}")

## while 的結束條件

`while 條件` 在每次進入區塊前檢查條件。區塊內必須使狀態往結束方向改變。
`*` 是乘法；`x *= 2` 將 x 乘二並寫回。

In [ ]:
# while：不知道要跑幾次時（例：加倍找 1000 以下最大的 2 的冪）
x = 1
while x * 2 <= 1000:
    x *= 2
print("1000 以下最大的 2 的冪：", x)

In [ ]:
# 練習 3：從 pairs 建出 dict，並算出「有成績的人數」（值不是 None 才算）
pairs = [("S001", 88), ("S002", None), ("S003", 74), ("S004", None)]

score_map = {}      # TODO：建 dict
n_scored = 0        # TODO：數非 None

print('練習 3：待核對', (score_map.get("S003"), n_scored), "；期望", (74, 2))

In [ ]:
# 練習 4：數出 grades 裡「及格（>=60）」的筆數與總分
grades = [88, 55, 74, 92, 61, 43]
n_pass = 0        # TODO
total_pass = 0    # TODO

print('練習 4：待核對', (n_pass, total_pass), "；期望", (4, 315))

## comprehension：篩選與轉換

```python
[運算式 for x in 來源 if 條件]
```
逐一取出來源值，保留條件成立者，再計算每個輸出值。
dict 版本寫成 `{鍵: 值 for ...}`，其結果仍是鍵值配對。

`min()`／`max()` 取最小／最大值；`+` 加法、`**` 次方，因此 `g ** 0.5` 是平方根。
`round(x)` 捨入整數，`round(x, n)` 捨入至 n 位小數；中間值採五成雙，浮點表示也可能影響結果。
`{i:03d}` 以三位十進位整數顯示，不足補零；`print(..., sep="\n")` 將各項分行顯示。

In [ ]:
grades = [88, 55, 74, 92, 61]

passed = [g for g in grades if g >= 60]                  # 篩選
curved = [min(g + 5, 100) for g in grades]               # 逐項計算
labels = ["及格" if g >= 60 else "不及格" for g in grades]
pair_dict = {f"S{i:03d}": g for i, g in enumerate(grades, 1)}   # dict 版

print(passed, curved, labels, sep="\n")
print(pair_dict)

In [ ]:
# 練習 5：一行 comprehension——取出「及格成績開根號 ×10」的清單（用 round 捨入成整數）
grades = [88, 55, 74, 92, 61]
curved_sqrt = []      # TODO：期望 [94, 86, 96, 78]（55 不及格被濾掉）

print('練習 5：待核對', curved_sqrt, "；期望", [94, 86, 96, 78])

## 函數：把「一件事」包起來

```python
def function_name(value, option=1):
    """docstring：一句話說明"""
    return value
```
把資料處理邏輯包在函數裡，就能用不同輸入直接測試，不必每次操作整個介面。

`def` 定義函數，參數接收輸入，`return` 回傳結果。呼叫時可省略有預設值的參數，也可按名稱指定。
空 list 在條件中視為 False；`not xs` 用來辨認空輸入。三引號是多行字串，函數首個字串是 docstring。
`lambda xs: 運算式` 建立單一回傳式的函數；`sorted(..., key=函數)` 依函數算出的值排序。
`key=lambda pair: -pair[1]` 使用配對第二值的負數，使較高分排在前面。
`abs()` 取絕對值。`...` 可作待完成函數的占位；沒有 return 的函數回傳 None，不代表已完成。

In [ ]:
def average(xs, ndigits=1):
    """回傳平均（預設一位小數）；空清單回 None"""
    if not xs:
        return None
    return round(sum(xs) / len(xs), ndigits)

print(average([88, 74, 92]))
print(average([88, 74, 92], ndigits=3))
print(average([]))

top2 = lambda xs: sorted(xs, reverse=True)[:2]      # lambda：一行的小函數（排序 key 最常用）
print("前二名：", top2([88, 55, 74, 92]))
print("按絕對值排：", sorted([-5, 3, -1], key=abs))

## 檢查器：比較實際結果與期望值

`check(got, expected, label)` 以 `==` 比較兩個結果，並回傳 True 或 False。
顯示文字使用已學的條件式；f-string 的 `!r` 顯示含引號等資訊的值表示，便於看出空字串與 None。

In [ ]:
# 檢查器：check(你算的, 正確答案, 題號)
def check(got, expected, label):
    ok = got == expected
    print(f"{label}：", "✅ 通過" if ok else f"未通過（目前得到 {got!r}）")
    return ok

print("檢查器就緒")

In [ ]:
# 可變預設參數：先寫下預期的兩次輸出，再執行
def add_student(name, roster=[]):
    roster.append(name)
    return roster

print(add_student("A"))
print(add_student("B"))   # 比較第一次和第二次的結果

# 預設參數只在「定義時」建立一次！修法：roster=None，函數內 if roster is None: roster = []

In [ ]:
# 練習 6：寫 clamp(x, lo, hi)——把 x 夾在 [lo, hi] 之間（低於取 lo、高於取 hi）
def clamp(x, lo=0, hi=100):
    ...      # TODO（提示：min 和 max 各用一次）

check((clamp(105), clamp(-3), clamp(88), clamp(7, lo=10)), (100, 0, 88, 10), "練習 6")

# 2 輸入、檔案與資料規則

將外部文字整理成可計算的成績，以函數檢查每列，分開保存接受資料與拒絕原因。

## 例外處理：區分無效輸入與有效結果

`try` 放可能失敗的操作，`except ErrorType as error` 處理指定類型並取得錯誤物件。
`int("88")` 將整數文字轉成整數；無法轉換的文字引發 ValueError，除以零引發 ZeroDivisionError。
`type(error).__name__` 取得例外的類型名稱。只接預期類型，其他錯誤仍向外傳出。
回傳 None 表示沒有有效結果，不能用 0 代替，以免混淆真正的零分。
`0 <= g <= 100` 同時檢查上下界，用來判斷轉換後的成績是否在範圍內。

In [ ]:
def safe_div(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None                     # 分母為零 → 回 None，讓呼叫者自己決定怎麼辦

print(safe_div(10, 4), safe_div(10, 0))

for raw in ["88", "八十八", ""]:
    try:
        print(f"{raw!r} → {int(raw)}")
    except ValueError as e:
        print(f"{raw!r} → 無法轉換（{type(e).__name__}: {e}）")

In [ ]:
# 練習 7：寫 parse_grade(s)——字串轉 int；轉不了回 None；成功但超出 0–100 也回 None
def parse_grade(s):
    ...      # TODO

check((parse_grade("88"), parse_grade("abc"), parse_grade("120"), parse_grade("-1")),
      (88, None, None, None), "練習 7")

### 錯誤訊息閱讀：先看最後一行，再往上追

程式出錯時，Python 會印出 **traceback**（呼叫路徑）。依以下順序定位：

1. **最後一行**：錯誤類型＋原因（例如 `ValueError: invalid literal ...`）。
2. **呼叫位置**：由末端往前找自己寫的函數與出錯行。
3. 再往上看：是誰一層層叫到這裡。

| 常見類型 | 先問自己 | 常見修法 |
|---|---|---|
| `NameError` | 名字是否拼錯？定義那格有沒有先跑？ | 修拼字／重跑前置 cell |
| `TypeError` | 值的型別適合這個運算嗎？ | 印出 `type(x)` |
| `KeyError` / `IndexError` | 這個鍵／位置真的存在嗎？ | 先印 keys／`len(xs)` |
| `ValueError` | 型別可以轉，但內容不合法嗎？ | 檢查原始值與邊界條件 |

向他人或 AI 提問時，一起提供錯誤類型、自己的出錯行與當時輸入。

`import traceback` 載入標準模組；`traceback.format_exc()` 在 except 內取得呼叫路徑文字。
`lambda: 運算式` 是不接參數的函數；`except (TypeError, KeyError, IndexError)` 接住列出的三類錯誤。

In [ ]:
# 刻意製造一個錯誤，但用 try/except 把證據保留下來，所以 notebook 不會中斷
import traceback

def parse_percent(raw):
    return round(int(raw) / 100, 2)

def import_one(row):
    return parse_percent(row["rate"])

try:
    import_one({"rate": "八成"})
except ValueError as err:
    trace_text = traceback.format_exc()
    print(trace_text)
    print("── 三步讀法 ──")
    print("1. 類型：", type(err).__name__)
    print("2. 原因：int() 不能把中文數字字串轉成整數")
    print("3. 現場：row['rate'] = '八成'；應改成數字輸入，或在邊界做明確驗證")

In [ ]:
# 閱讀練習：先判斷每案的錯誤類型，再執行，並說明輸入值的問題
error_cases = [
    ("數字與字串相加", lambda: 3 + "4"),
    ("缺少欄位", lambda: {"sid": "S001"}["grade"]),
    ("位置超出清單", lambda: [88][3]),
]

for label, action in error_cases:
    try:
        action()
    except (TypeError, KeyError, IndexError) as err:
        print(f"{label:<10} → {type(err).__name__}: {err}")

print("\n請指出每案應檢查哪個中間值，才能驗證原因。")

## 檔案與 with：寫入後關閉

`open(path, "w", encoding="utf-8")` 以 UTF-8 寫文字；w 會清空同名檔，故此例只寫新暫存目錄。
`with ... as file` 使離開區塊時關閉檔案。`.write()` 寫文字；`\n` 是換行。
省略模式時讀取檔案，`for line in file` 逐行讀取，`.rstrip()` 去掉尾端空白與換行。
其他物件的 with 行為須依該物件的介面判斷，不能一律理解為關檔。

## 檔案例子的操作位置

`from pathlib import Path` 將 pathlib 的 Path 名稱匯入目前程式；Path 用來建立路徑物件。
`路徑 / "檔名"` 組合子路徑。
`tempfile.mkdtemp()` 建立新的暫存目錄。每次從頭執行都取得新目錄，不覆寫你原有的 note.txt。

In [ ]:
from pathlib import Path
import tempfile

python_dir = Path(tempfile.mkdtemp(prefix="db-u01-python-"))
note_path = python_dir / "note.txt"

with open(note_path, "w", encoding="utf-8") as f:
    f.write("第一行：Python\n第二行：學生資料\n")

with open(note_path, encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        print(f"{i}: {line.rstrip()}")

## CSV：先解析欄位，再轉型

csv 與 io 是 Python 標準模組。`io.StringIO(text)` 把文字包成可讀取的記憶體檔案，不寫入硬碟。
`csv.DictReader(file)` 用第一列作欄名，其餘每列是 dict，儲存格先是文字。
成績須先用 int 轉成整數才能加五，不能把文字「88」直接當數值運算。

In [ ]:
import csv, io
raw_csv = io.StringIO("sid,grade\nS001,88\nS002,55\n")       # 以記憶體文字作 CSV 輸入
for row in csv.DictReader(raw_csv):
    print(row["sid"], int(row["grade"]) + 5)

## class：把資料與操作放在同一物件

`class GradeBook` 定義物件型態。呼叫 GradeBook() 時執行 `__init__`；self 表示該物件。
每個物件的 `self.data` 保存自己的 dict，方法透過 self 讀寫它。
`setdefault(key, [])` 在缺鍵時放入新 list，並回傳該鍵目前的值；接 `.append(score)` 加入分數。
此例的呼叫者提供數值分數，沒有包含外部輸入驗證。

In [ ]:
class GradeBook:
    def __init__(self):                     # 建構：初始化自己的資料
        self.data = {}                      # self = 這一本 GradeBook 自己

    def add(self, sid, score):
        self.data.setdefault(sid, []).append(score)

    def average(self, sid):
        xs = self.data.get(sid)
        return sum(xs) / len(xs) if xs else None

gb = GradeBook()                            # 造一本（呼叫 __init__）
gb.add("S001", 80)
gb.add("S001", 90)
gb.add("S002", 70)
print(gb.average("S001"), gb.average("S999"))
print("gb.add 與 gb.average 都以物件.方法(參數)呼叫。")

## 模組與套件

`import statistics` 匯入模組；`from collections import Counter` 將 collections 的 Counter 名稱匯入目前程式。
statistics 與 collections 是 Python 標準函式庫，不須另外安裝。
第三方套件則需安裝到目前 Python 環境；pip 是管理這些套件的工具。

`statistics.mean`、`median`、`stdev` 分別計算平均、中位數、樣本標準差。
`Counter(values).most_common(2)` 回傳出現次數最多的兩個值及次數。

In [ ]:
import statistics
from collections import Counter

grades = [88, 55, 74, 92, 61, 74]
print("平均", statistics.mean(grades), "／中位數", statistics.median(grades),
      "／標準差", round(statistics.stdev(grades), 2))
print("次數統計：", Counter(grades).most_common(2))
# 本格只使用 Python 標準函式庫。

### 練習 1–7 參考解答

各題解答使用該題原先設定的輸入值；請在對應工作格執行。

<details><summary>展開參考解答</summary>

```python
# 練習 1：f-string
msg = f"{sid} 的 {n_course} 科平均是 {score:.1f} 分"
# 練習 2：list 與切片
trimmed = sorted(grades)[1:-1]
# 練習 3：dict、tuple、set
score_map = {sid: g for sid, g in pairs}
n_scored = len([g for _, g in pairs if g is not None])
# 練習 4：流程控制
n_pass = len([g for g in grades if g >= 60])
total_pass = sum([g for g in grades if g >= 60])
# 練習 5：comprehension
curved_sqrt = [round(g ** 0.5 * 10) for g in grades if g >= 60]
# 練習 6：函數
def clamp(x, lo=0, hi=100):
    return max(lo, min(x, hi))
# 練習 7：例外處理
def parse_grade(s):
    try:
        g = int(s)
    except ValueError:
        return None
    return g if 0 <= g <= 100 else None
```
</details>

## assert：固定預期結果

`assert 條件` 在條件不成立時引發 AssertionError，使執行停止。
以下分別核對長度、總分與每個成績值。

In [ ]:
original_grades = [88, 55, 74]
assert len(original_grades) == 3
assert sum(original_grades) == 217
assert original_grades == [88, 55, 74]

## 共用清單與複製清單

`alias = original_grades` 只多一個名稱，不會複製內容。
`list(original_grades)` 建立新清單，但其中若還有可變物件，仍須考慮內層是否共用。
純量成績可以用這種方式複製；改副本不應改到原成績。

In [ ]:
original_grades = [88, 55, 74]
alias = original_grades
alias.append(92)
assert original_grades == [88, 55, 74, 92]
copied_grades = list(original_grades)
copied_grades[0] = 0
assert original_grades[0] == 88
assert copied_grades[0] == 0

## 整理文字，不修改原始輸入

`.strip()` 去掉文字兩端空白；`.upper()` 將英文字母改大寫。
字串操作回傳新字串，原字串不改變。先定義學號格式，再選擇要做的轉換。
本例把學號首尾空白及大小寫差異視為相同識別碼，不移除中間字元。

In [ ]:
raw_sid = " s001 "
clean_sid = raw_sid.strip().upper()
assert raw_sid == " s001 "
assert clean_sid == "S001"
assert "林 佳蓉".strip() == "林 佳蓉"

## 將成績匯入規則寫成函數

每列須提供文字型別的 sid、name、grade；學號及姓名不能只有空白，成績須能轉為 0–100 整數。
`type(value) is str` 檢查型別是否正好是字串，`is not str` 表示相反。
空字串在條件中為 False，因此 `not sid` 可以找出空學號。
`raise ValueError("原因")` 主動拒絕違反規則的輸入。`str(error)` 取得錯誤文字。
函數回傳新的 dict，不在來源列上修改。先完成每列驗證，再處理重複學號。

In [ ]:
def clean_record(raw):
    for field in ["sid", "name", "grade"]:
        if type(raw.get(field)) is not str:
            raise ValueError(f"{field} 必須提供文字")
    sid = raw["sid"].strip().upper()
    name = raw["name"].strip()
    if not sid or not name:
        raise ValueError("學號與姓名不能空白")
    grade = int(raw["grade"].strip())
    if not 0 <= grade <= 100:
        raise ValueError("成績須介於 0–100")
    return {"sid": sid, "name": name, "grade": grade}

assert clean_record({"sid": " s001 ", "name": " 林佳蓉 ", "grade": "88"}) == {
    "sid": "S001", "name": "林佳蓉", "grade": 88}

## 區分缺值、零分與不合法文字

0 在條件判斷中視為 False，但在本例是合法成績，不能因此略過。
缺少 grade、None、文字「八十」及 101 都不符合本例規則。

In [ ]:
assert clean_record({"sid": "S005", "name": "零分", "grade": "0"})["grade"] == 0
for raw in [
    {"sid": "S003", "name": "測試"},
    {"sid": "S003", "name": "測試", "grade": None},
    {"sid": "S003", "name": "測試", "grade": "八十"},
    {"sid": "S003", "name": "測試", "grade": "101"},
]:
    rejected = False
    try:
        clean_record(raw)
    except ValueError as error:
        rejected = True
        print(str(error))
    assert rejected

## CSV 原始資料也要保留下來

用已學過的 StringIO 與 DictReader 讀取文字；每列的 grade 仍是字串。
`dict(row)` 建立一列的新 dict；本例各值都是文字，可以逐列複製後核對輸入是否被改動。
此批次含有效、非法、重複與空白資料。第二次 S001 不可直接覆寫第一筆。
原始紀錄保持不變，讓拒絕原因可以對回來源列。

In [ ]:
import csv
import io

raw_text = "sid,name,grade\nS001,林佳蓉,88\ns002,陳威廷,55\nS003,吳孟軒,八十\nS001,林佳蓉,93\nS004, ,91\nS005,零分,0\n"
source_rows = list(csv.DictReader(io.StringIO(raw_text)))
source_before = [dict(row) for row in source_rows]
assert len(source_rows) == 6
assert source_rows[0]["grade"] == "88"
print(source_rows)

## 每列都有去向：接受或拒絕

accepted 保存清理成功的列；rejected 保存來源位置與原因；seen 保存已接受的學號。
`set.add(value)` 將值加入集合，相同值不會重複保存。
`continue` 略過本輪的其餘程式，繼續下一列。只有成功接受後才加入 seen，非法資料不占用學號。
`enumerate(..., start=2)` 把標題列算作第 1 列，讓位置對應本例沒有欄內換行的 CSV。

In [ ]:
def clean_records(rows):
    accepted, rejected, seen = [], [], set()
    for position, raw in enumerate(rows, start=2):
        try:
            record = clean_record(raw)
        except ValueError as error:
            rejected.append({"position": position, "reason": str(error)})
            continue
        if record["sid"] in seen:
            rejected.append({"position": position, "reason": "重複學號"})
            continue
        accepted.append(record)
        seen.add(record["sid"])
    return accepted, rejected

## 核對接受、拒絕及輸入不變

六筆來源中接受 S001、S002、S005；其餘有明確拒絕原因。
核對每個識別碼、成績與位置，再確認來源列未被修改。

In [ ]:
accepted, rejected = clean_records(source_rows)
assert [row["sid"] for row in accepted] == ["S001", "S002", "S005"]
assert [row["grade"] for row in accepted] == [88, 55, 0]
assert [row["position"] for row in rejected] == [4, 5, 6]
assert len(accepted) + len(rejected) == len(source_rows)
assert source_rows == source_before
assert clean_records([]) == ([], [])
print("接受：", accepted)
print("拒絕：", rejected)

## 把接受資料寫入 CSV

`csv.DictWriter(file, fieldnames=...)` 依指定欄位順序寫 dict；writeheader 寫標題，writerows 寫多列。
開 CSV 檔時使用 `newline=""`，由 csv 模組處理列分隔；有逗號或換行的值也交由它處理引號。
檔案寫入已建立的 python_dir。

In [ ]:
accepted_path = python_dir / "accepted.csv"
with open(accepted_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["sid", "name", "grade"])
    writer.writeheader()
    writer.writerows(accepted)
print(accepted_path)

## 重讀檔案後再核對完整內容

寫入數值後重新讀 CSV，儲存格仍是文字；要依相同規則轉型後再比較。
檔名存在、筆數相同或總分相同，都不足以證明每列未變。
這裡按原順序逐列比較 sid、name、grade。

In [ ]:
with open(accepted_path, encoding="utf-8", newline="") as file:
    reloaded = [clean_record(row) for row in csv.DictReader(file)]
assert reloaded == accepted
changed_copy = [dict(row) for row in reloaded]
changed_copy[0]["grade"] = 0
assert changed_copy != accepted
assert accepted[0]["grade"] == 88

## 把資料整理成明確的成績報告

報告保存接受筆數、拒絕筆數與平均；平均的分母只有接受資料。
回傳值固定是 dict；沒有接受資料時，average 為 None，兩個筆數欄位仍保留。
清理與統計分成不同函數，讓不同批次重用同一計算規則。

In [ ]:
def build_report(valid_rows, rejected_rows):
    grades = [row["grade"] for row in valid_rows]
    return {"accepted_count": len(valid_rows), "rejected_count": len(rejected_rows),
            "average": round(sum(grades) / len(grades), 1) if grades else None}

report = build_report(accepted, rejected)
assert report == {"accepted_count": 3, "rejected_count": 3, "average": 47.7}
assert build_report([], []) == {"accepted_count": 0, "rejected_count": 0, "average": None}
print(report)

## 用 JSON 保存報告

JSON 可以保存物件、陣列、文字、數值、布林及空值，對應本例的 dict、list、str、數值、bool 與 None。
標準模組 json 的 `dump(value, file)` 寫出資料，`load(file)` 讀回；ensure_ascii=False 讓中文字直接保存。
`indent=2` 用兩個空白縮排，讓存出的文字較容易閱讀。
JSON 報告保存已計算結果；來源資料與拒絕紀錄仍需另外保存，不能由平均值還原。

In [ ]:
import json

report_path = python_dir / "report.json"
with open(report_path, "w", encoding="utf-8") as file:
    json.dump(report, file, ensure_ascii=False, indent=2)
with open(report_path, encoding="utf-8") as file:
    loaded_report = json.load(file)
assert loaded_report == report
print(loaded_report)

# 3 資料處理、分析與圖表

NumPy 處理數值陣列，pandas 整理有欄名的表格，matplotlib 繪圖。
`import 套件 as 簡稱` 讓後續程式以 np、pd、plt 呼叫這三個套件。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("已載入 NumPy、pandas、matplotlib")

## ndarray：同型別的數值陣列

`np.array(清單)` 建立 ndarray；`.shape` 是各維度的長度，`.dtype` 是元素型別。
`(5,)` 表示一維、五個元素。`xs + 5` 逐項加五，不會串接清單。
建立新陣列不會修改原 list。

In [ ]:
xs = np.array([88, 55, 74, 92, 61])
print("形狀：", xs.shape, "型別：", xs.dtype)
print("整條陣列一起算：", xs + 5)

## 陣列索引、切片與副本

`xs[0]` 取第一個值；`xs[1:3]` 取位置 1、2。
NumPy 的基本切片與原陣列共用資料。修改切片中的值，也會修改原陣列。
`.copy()` 建立獨立副本；本例只修改副本，保留原成績。

In [ ]:
print(xs[0], xs[1:3])
adjusted = xs.copy()
adjusted[1:3] = adjusted[1:3] + 5
print("原值：", xs)
print("修改副本：", adjusted)
assert xs[1] == 55
assert adjusted[1] == 60

## 布林遮罩：按條件取值

`xs >= 60` 逐項產生布林值；`xs[遮罩]` 只取 True 對應的位置。
陣列條件以 `&` 表示逐項且、`|` 表示逐項或，各比較式都加括號。
不要用 Python 的 and、or 合併整個陣列。

In [ ]:
passing_mask = xs >= 60
print("及格位置：", passing_mask)
print("及格分數：", xs[passing_mask])
print("60 到 89 分：", xs[(xs >= 60) & (xs < 90)])
print("不及格或 90 分以上：", xs[(xs < 60) | (xs >= 90)])

## 平均、標準差與比例的分母

`.mean()` 算平均，`.std()` 預設用 N 為分母；`std(ddof=1)` 則用 N−1。
因此預設 std 不同於 statistics.stdev 的樣本標準差。
布林值 True 當作 1、False 當作 0，布林陣列的平均是符合條件的比例。

In [ ]:
print(f"平均 {xs.mean():.1f}、標準差 {xs.std():.2f}、及格率 {(xs >= 60).mean():.0%}")
print("樣本標準差：", xs.std(ddof=1))
assert (xs >= 60).mean() == 0.8

## 二維陣列：axis 決定沿哪個方向計算

本例每列是一名學生、每欄是一場考試。`matrix[:, 0]` 取全部列的第一欄。
`mean(axis=0)` 對列方向合併，得到每欄平均；`mean(axis=1)` 得到每列平均。
`np.array_equal(a, b)` 檢查兩陣列的形狀與值相同。

In [ ]:
matrix = np.array([[80, 90], [60, 70], [100, 80]])
print(matrix, "形狀：", matrix.shape)
print("第一場考試：", matrix[:, 0])
print("各場考試平均：", matrix.mean(axis=0))
print("各學生平均：", matrix.mean(axis=1))
assert np.array_equal(matrix.mean(axis=1), np.array([85, 65, 90]))

## 缺值不是零分

`np.nan` 表示缺少浮點數值；一般整數陣列不能直接容納 NaN。
`np.isnan(values)` 判斷缺值；不能用 `values == np.nan` 判斷。
普通 mean 遇到 NaN 會得到 NaN。`np.nanmean` 略過 NaN，分母只計非缺值。

In [ ]:
incomplete = np.array([80.0, np.nan, 0.0])
print("缺值位置：", np.isnan(incomplete))
print("一般平均：", incomplete.mean())
print("非缺值平均：", np.nanmean(incomplete))
assert np.isnan(incomplete.mean())
assert np.nanmean(incomplete) == 40

## 固定種子與重新產生

`np.random.default_rng(42)` 建立亂數產生器；`integers(0, 101, size=5)` 產生五個 0–100 整數。
下界包含、上界不包含。同一物件再次呼叫會往後取值，不是從頭重播。
在相同環境中重新建立相同種子的產生器，並依相同順序呼叫，才可重現。

In [ ]:
first_rng = np.random.default_rng(42)
first_draw = first_rng.integers(0, 101, size=5)
next_draw = first_rng.integers(0, 101, size=5)
replay_rng = np.random.default_rng(42)
replay = replay_rng.integers(0, 101, size=5)
print("第一次：", first_draw, "下一次：", next_draw)
assert np.array_equal(first_draw, replay)

## 模擬分布與合法範圍是兩件事

`normal(70, 10, 100_000)` 抽十萬個常態樣本；70、10 是分布的平均與標準差，不是上下界。
`.round()` 取到整數位，正好一半取最近偶數，但結果仍可為浮點陣列；`np.percentile(sim, 90)` 求第 90 百分位。
`np.clip(sim, 0, 100)` 將越界值截到邊界，會改變分布；不得把截斷當作未加工樣本。
布林陣列的 `.sum()` 數 True 的個數，可用來計算越界筆數。

In [ ]:
rng = np.random.default_rng(42)
sim = rng.normal(70, 10, 100_000).round()
print(f"模擬 {len(sim):,} 筆：平均 {sim.mean():.2f}、P90 = {np.percentile(sim, 90):.0f}")
bounded_sim = np.clip(sim, 0, 100)
print("原樣本越界筆數：", ((sim < 0) | (sim > 100)).sum())
assert ((bounded_sim >= 0) & (bounded_sim <= 100)).mean() == 1

## DataFrame：用欄名處理表格

`pd.DataFrame(dict)` 的鍵是欄名，每個 list 是一欄，各欄長度須相同。
`.shape` 是列數、欄數；`.dtypes` 顯示各欄型別；`df["grade"]` 是一欄 Series。
以下五人的 grade 是這張表自己的資料，不與前面的數值陣列逐項對應。

In [ ]:
df = pd.DataFrame({
    "sid":  ["S001", "S002", "S003", "S004", "S005"],
    "dept": ["統計", "統計", "資訊", "數學", "統計"],
    "grade": [88, 55, 93, 74, 61],
})
print(df)
print(df.shape)
print(df.dtypes)

## 明確指定文字與可缺值整數

`.astype({欄名: 型別})` 依欄轉型，回傳新表。
pandas 的 `"string"` 是文字型別；`"Int64"` 是可包含缺值的整數型別，大小寫有意義。
型別不能代替資料規則：成功轉為整數，仍不代表成績在 0–100。

In [ ]:
typed_df = df.astype({"sid": "string", "dept": "string", "grade": "Int64"})
print(typed_df.dtypes)
print(typed_df)
assert len(typed_df) == 5

## 指定欄位、列標籤與位置

`df[["sid", "grade"]]` 依給定順序取多欄；`.loc[列標籤, 欄名]` 按標籤取值。
`.iloc[列位置, 欄位置]` 按從零開始的位置取值；iloc 切片的右界不包含。
`.tolist()` 將一欄轉成普通 list。本例預設列標籤恰好也是 0–4，但標籤與位置不是同一概念。

In [ ]:
print(df[["sid", "grade"]])
print("標籤 2 的成績：", df.loc[2, "grade"])
print("前兩列、前兩欄：")
print(df.iloc[:2, :2])
assert df["sid"].tolist() == ["S001", "S002", "S003", "S004", "S005"]

## 按列篩選：遮罩須與資料列正確對應

`df.loc[條件, 欄名清單]` 同時指定列與欄。
Series 的多條件也使用有括號的 `&`、`|`；遮罩的索引會與表格索引對齊。
遮罩不必出自同一張表，但索引必須能正確對應；本例直接由 df 產生。

In [ ]:
stat_passing = (df["dept"] == "統計") & (df["grade"] >= 60)
selected_scores = df.loc[stat_passing, ["sid", "grade"]]
print(selected_scores)
assert selected_scores["sid"].tolist() == ["S001", "S005"]

## 排序與前幾筆

`.sort_values(["grade", "sid"], ascending=[False, True])` 先按分數降冪，再以學號升冪打破同分。
`.head(2)` 取前兩列。排序回傳新表，沒有重排原 df。
`na_position="last"` 明確把缺值放最後；取前幾筆之前，須先決定排序規則。

In [ ]:
ranked = df.sort_values(["grade", "sid"], ascending=[False, True], na_position="last")
print(ranked.head(2))
assert ranked.head(2)["sid"].tolist() == ["S003", "S001"]
assert df["sid"].tolist()[0] == "S001"

## 保留缺值，再決定處理方式

`.copy()` 建立獨立表；`pd.NA` 可表示 pandas 可缺值型別中的缺值。
`.isna()`、`.notna()` 判斷缺值與非缺值；`.sum()` 可計算布林 True 的個數。
pandas 的 mean 預設略過缺值。`.fillna(0)` 會把缺值改成零；只有規則確定缺值代表零時才適用。
非缺值不代表符合成績規則；999 也會被 notna 與 mean 納入。

In [ ]:
missing_df = typed_df.copy()
missing_df.loc[1, "grade"] = pd.NA
print(missing_df)
print("缺值數：", missing_df["grade"].isna().sum())
print("非缺值平均：", missing_df["grade"].mean())
print("誤把缺值當零的平均：", missing_df["grade"].fillna(0).mean())
assert missing_df["grade"].notna().sum() == 4
assert typed_df.loc[1, "grade"] == 55

## 分組後的列數、非缺值數與平均

`groupby("dept", as_index=False)` 按系所分組，系所保留為普通欄。
`agg(新欄名=(來源欄, 運算))` 命名計算結果：size 數列、count 數非缺值、mean 算非缺值平均。
分組鍵若缺值，預設不納入分組；本例先確認系所沒有缺值。

In [ ]:
assert missing_df["dept"].isna().sum() == 0
grade_summary = missing_df.groupby("dept", as_index=False).agg(
    n=("grade", "size"), nonmissing_n=("grade", "count"), avg_grade=("grade", "mean"))
print(grade_summary)
stat_summary = grade_summary.loc[grade_summary["dept"] == "統計"]
assert stat_summary["n"].tolist() == [3]
assert stat_summary["nonmissing_n"].tolist() == [2]
assert stat_summary["avg_grade"].tolist() == [74.5]

## 資料列的身分：先決定什麼不能重複

`.duplicated(欄名清單, keep=False)` 標記所有具有重複鍵值的列。
`.any()` 判斷至少一個 True，`.all()` 判斷是否全部 True。
五人名冊以 sid 識別一人；同一人的不同學期成績則不能只以 sid 判定重複。

In [ ]:
repeated_students = df.duplicated(["sid"], keep=False)
print("重複學號列：")
print(df.loc[repeated_students])
assert not repeated_students.any()
assert df["grade"].notna().all()

## concat：兩學期成績上下相接

`pd.concat([fall_scores, spring_scores], ignore_index=True)` 上下合併，並重建列標籤。
concat 不刪除重複資料，也不檢查 sid、semester 的組合是否唯一。
一列代表一名學生在某學期的成績。

In [ ]:
fall_scores = pd.DataFrame({
    "sid": ["S001", "S002"], "semester": ["113-1", "113-1"], "grade": [88, 55]})
spring_scores = pd.DataFrame({
    "sid": ["S001", "S003"], "semester": ["113-2", "113-2"], "grade": [91, 74]})
all_scores = pd.concat([fall_scores, spring_scores], ignore_index=True)
print(all_scores)
assert all_scores.shape == (4, 3)
assert not all_scores.duplicated(["sid", "semester"]).any()

## merge：依學號補上系所

`.merge(student_lookup, on="sid", how="left")` 按 sid 配對，保留左表成績。
`validate="many_to_one"` 要求右表每個 sid 唯一；左邊同一學生可以有多學期成績。
`.to_string(index=False)` 顯示表格時省略列標籤。本例配對後應仍有四列，且系所沒有缺值。

In [ ]:
student_lookup = pd.DataFrame({
    "sid": ["S001", "S002", "S003"], "dept": ["統計", "統計", "資訊"]})
score_detail = all_scores.merge(
    student_lookup, on="sid", how="left", validate="many_to_one")
print("concat 後：", all_scores.shape, "（4 列 × 3 欄）")
print("merge 後：", score_detail.shape, "（列數不變、接上 dept）")
print(score_detail.to_string(index=False))
assert score_detail.shape == (4, 4)
assert score_detail["dept"].isna().sum() == 0

## 重複右鍵會把一筆成績展開成多筆

`student_lookup.iloc[[0]]` 以位置清單取回一列 DataFrame，而不是單一值。
本例刻意追加 S001 的重複名冊列。未驗證的配對會增加筆數；已驗證的配對引發 `pd.errors.MergeError`。
rejected_duplicate 記錄這個例外是否確實發生。

In [ ]:
repeated_lookup = pd.concat([student_lookup, student_lookup.iloc[[0]]], ignore_index=True)
expanded = all_scores.merge(repeated_lookup, on="sid", how="left")
print("原列數：", len(all_scores), "錯誤配對後：", len(expanded))
rejected_duplicate = False
try:
    all_scores.merge(repeated_lookup, on="sid", how="left", validate="many_to_one")
except pd.errors.MergeError as error:
    print(type(error).__name__, error)
    rejected_duplicate = True
assert len(expanded) == 6
assert rejected_duplicate

## 缺少對照資料，列數可能仍然正確

left 配對會保留找不到名冊的成績，dept 因而缺值。
validate 檢查鍵的重複關係，不保證每個左鍵都有對應。
配對後用 dept.isna() 找出缺少系所資料的成績；原因可能是找不到學號，也可能是名冊本身缺少系所。

In [ ]:
incomplete_lookup = student_lookup.loc[student_lookup["sid"] != "S003"]
unmatched = all_scores.merge(incomplete_lookup, on="sid", how="left", validate="many_to_one")
missing_students = unmatched.loc[unmatched["dept"].isna(), ["sid", "semester"]]
print(missing_students)
assert len(unmatched) == len(all_scores)
assert missing_students["sid"].tolist() == ["S003"]

## 配對前驗證學號

pandas 可能把兩側缺少鍵值的列互相配對；不能把這種結果當作找到同一名學生。
兩側 sid 都須為非空白字串，名冊 sid 不重複。
將兩欄分別用 `.tolist()` 取出，逐一以 `type(sid) is not str` 拒絕非字串，再以 `not sid.strip()` 拒絕空白。
`.strip()` 去掉文字兩端空白；本函數只檢查，不將清理後的值寫回輸入。
學號的大小寫與首尾空白統一，仍由前面的輸入整理負責。
一欄 Series 的 `.duplicated()` 標記與前面相同的值，再接 any 判斷是否出現重複。

In [ ]:
def validate_student_keys(score_rows, lookup):
    if score_rows["sid"].isna().any() or lookup["sid"].isna().any():
        raise ValueError("學號不得缺值")
    for sid_column in [score_rows["sid"], lookup["sid"]]:
        for sid in sid_column.tolist():
            if type(sid) is not str:
                raise ValueError("學號須為字串")
            if not sid.strip():
                raise ValueError("學號不得為空字串或全空白")
    if lookup["sid"].duplicated().any():
        raise ValueError("名冊學號不得重複")

validate_student_keys(all_scores, student_lookup)
missing_key_scores = all_scores.copy()
missing_key_scores.loc[0, "sid"] = None
rejected_missing_key = False
try:
    validate_student_keys(missing_key_scores, student_lookup)
except ValueError as error:
    print(error)
    rejected_missing_key = True
assert rejected_missing_key

In [ ]:
# 左表與名冊都要拒絕空字串、全空白及非字串學號。
for bad_sid in ["", "   ", 123, True]:
    bad_key_scores = pd.DataFrame({"sid": [bad_sid], "semester": ["113-1"], "grade": [88]})
    bad_key_lookup = pd.DataFrame({"sid": [bad_sid], "dept": ["統計"]})
    for left, right in [(bad_key_scores, student_lookup), (all_scores, bad_key_lookup)]:
        rejected_key = False
        try:
            validate_student_keys(left, right)
        except ValueError as error:
            print(f"{bad_sid!r}", error)
            rejected_key = True
        assert rejected_key

## 計算新欄，再交給一般 Python 程式

`表["新欄名"] = 運算結果` 新增欄位；兩欄相乘會按相同列標籤配對計算。
`.itertuples(index=False, name=None)` 逐列產生普通 tuple，不加入表格列標籤。
外面用 list 收集，便能以已學過的 check 核對結果。

In [ ]:
adjusted_detail = score_detail.copy()
adjusted_detail["adjusted_grade"] = adjusted_detail["grade"] + 5
score_pairs = list(adjusted_detail[["sid", "adjusted_grade"]].itertuples(index=False, name=None))
print(score_pairs)
assert score_pairs == [("S001", 93), ("S002", 60), ("S001", 96), ("S003", 79)]

## CSV 輸出的檔案位置

本節將五人資料寫入 grades.csv，學號型別示例寫入 numeric_ids.csv。
data_dir 是本節輸出目錄，data_csv_path 指向其中的 grades.csv。

In [ ]:
from pathlib import Path
import tempfile

data_dir = Path(tempfile.mkdtemp(prefix="u01-data-"))
data_csv_path = data_dir / "grades.csv"
print("輸出目錄：", data_dir)

## CSV 沒有替你保存欄位型別

`.to_csv(path, index=False, encoding="utf-8")` 寫出欄名與資料，不另存列標籤。
`pd.read_csv(path, dtype={...})` 讀檔時指定型別；全數字學號若未指定文字，前導零可能消失。
S001 不易被誤判，因此用 0001、0010 觀察前導零是否保留。

In [ ]:
numeric_ids = pd.DataFrame({"sid": ["0001", "0010"], "grade": [88, 55]})
ids_path = data_dir / "numeric_ids.csv"
numeric_ids.to_csv(ids_path, index=False, encoding="utf-8")
guessed_ids = pd.read_csv(ids_path)
preserved_ids = pd.read_csv(ids_path, dtype={"sid": "string", "grade": "Int64"})
print("自動推測：", guessed_ids["sid"].tolist())
print("指定文字：", preserved_ids["sid"].tolist())
assert preserved_ids["sid"].tolist() == ["0001", "0010"]

## 寫出再重讀，核對資料而不是只檢查檔案存在

`.equals(other)` 核對表格形狀、列欄標籤、欄位型別與內容；同位置缺值可視為相同。
以下把含缺值的五人資料寫成 CSV，再按相同型別重讀。
read_csv 預設會將空欄位等指定標記解讀為缺值；若 NA 是合法學號文字，需另外設定缺值規則。

In [ ]:
missing_df.to_csv(data_csv_path, index=False, encoding="utf-8")
reread_df = pd.read_csv(data_csv_path, dtype={"sid": "string", "dept": "string", "grade": "Int64"})
print(reread_df)
assert reread_df.equals(missing_df)
assert reread_df["grade"].isna().sum() == 1

## 圖表先確定每個位置代表什麼

`plt.subplots(1, 2, figsize=(10, 3))` 建立一張圖及左右兩個座標區；axes[0]、axes[1] 各選一區。
`.plot(x, y, marker="o")` 依輸入次序連線；`.bar(名稱, 數值)` 畫長條，兩個清單長度須相同。
`set_title`、`set_ylabel` 設定標題與縱軸。圖中文字用英文，資料中的中文仍保留。

## 一格完成建圖、顯示與關閉

`plt.tight_layout()` 調整間距，`plt.show()` 顯示圖，`plt.close(fig)` 釋放圖形資源。
以下兩圖使用示意數值：學期平均 71.2、72.8、70.5、73.9；各系人數 8、4、4、4。
這些是獨立示例，不是前面五人資料或四筆學生學期成績的計算結果。

In [ ]:
sems = ["113-1", "113-2", "114-1", "114-2"]
avgs = [71.2, 72.8, 70.5, 73.9]
dept_names = ["統計", "資訊", "數學", "企管"]
dept_counts = [8, 4, 4, 4]
english_depts = ["Statistics", "CS", "Math", "Business"]
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(sems, avgs, marker="o")
axes[0].set_title("Semester mean")
axes[0].set_ylabel("Grade")
axes[1].bar(english_depts, dept_counts)
axes[1].set_title("Student count")
plt.tight_layout()
plt.show()
plt.close(fig)

## 圖中數字來自實際整理結果

`.map(dict)` 依對照表替換一欄文字；沒有對應的值會變成缺值，須檢查。
以下由四筆學生學期成績計算各學期平均與各系紀錄數；同一學生跨學期會計為多筆，紀錄數不是不重複學生數。
`.savefig(path)` 保存這張圖；在 show、close 之前儲存，不依賴上一格還留著圖形。

In [ ]:
semester_report = all_scores.groupby("semester", as_index=False).agg(avg_grade=("grade", "mean"))
semester_report = semester_report.sort_values("semester")
dept_report = score_detail.groupby("dept", as_index=False).agg(n=("grade", "size"))
dept_report = dept_report.sort_values("dept")
dept_labels = dept_report["dept"].map({"統計": "Statistics", "資訊": "CS"})
assert dept_labels.notna().all()
print(semester_report)
print(dept_report)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(semester_report["semester"], semester_report["avg_grade"], marker="o")
axes[0].set_title("Mean of recorded grades")
axes[0].set_ylabel("Grade")
axes[1].bar(dept_labels, dept_report["n"])
axes[1].set_title("Student-semester records")
plt.tight_layout()
fig.savefig(data_dir / "score_report.png")
plt.show()
plt.close(fig)
assert semester_report["avg_grade"].tolist() == [71.5, 82.5]
assert dept_report["n"].tolist() == [3, 1]

## 平均的分母：非缺值筆數

新增一筆學生學期成績，但分數缺值。系所紀錄數增加，非缺值成績數與平均則不變。
Series 的 `.round(2)` 產生捨入至小數第二位的數值；指定回欄位會改變數值，不只是顯示格式，也不增加測量精度。
n 計全部紀錄，nonmissing_n 計非缺值成績，avg_grade 是非缺值平均；這些計算不會驗證成績範圍。

In [ ]:
report_scores = pd.concat([all_scores, pd.DataFrame({
    "sid": ["S002"], "semester": ["114-1"], "grade": [np.nan]})], ignore_index=True)
report_detail = report_scores.merge(student_lookup, on="sid", how="left", validate="many_to_one")
observed_report = report_detail.groupby("dept", as_index=False).agg(
    n=("grade", "size"), nonmissing_n=("grade", "count"), avg_grade=("grade", "mean"))
observed_report["avg_grade"] = observed_report["avg_grade"].round(2)
print(observed_report)
assert observed_report["n"].tolist() == [4, 1]
assert observed_report["nonmissing_n"].tolist() == [3, 1]
assert observed_report["avg_grade"].tolist() == [78.0, 74.0]

## 檔案、分析工具與 DBMS 的分工

| 工具 | 本課已能辨認的用途 |
|---|---|
| CSV、JSON | 交換及保存資料；讀寫程式決定型別與驗證方式 |
| NumPy、pandas | 在程式中計算、清理、拼接與分組資料 |
| matplotlib | 把整理出的結果畫成圖表 |
| DBMS | 管理共用資料的儲存、讀取、規則與修改 |

這些工具可以分工合作。分析時取得的副本不會自動跟隨來源更新，圖表也不是資料備份。
一次性的 CSV 統計不必為此更換工具；若多個人反覆修改同一組正式資料，就須安排統一的修改管理。

## 什麼情況繼續用 Excel 或 pandas？

少量資料需要人工檢視或編輯，可以用 Excel 等試算表；固定的清理、計算與圖表流程，可以用 pandas 程式重複執行。
分析一份不再修改的 CSV，不必只為了「使用資料庫」而搬移資料。

若多個程式會反覆修改正式資料、需要限制操作或統一檢查規則，就應評估 DBMS 如何配合應用程式。
不應只以「多人」或固定筆數畫界線；還要確認工具如何協調修改、資料格式是否一致，以及誰負責維護。

問題：一份唯讀 CSV 要計算系所平均並畫圖，是否一定需要 PostgreSQL？

<details><summary>參考方向</summary>

不一定。可先用已學過的 pandas 與 matplotlib 完成；資料量或執行成本成為問題時，再評估 DuckDB 等工具。
目前需求沒有要求多個程式共同修改，不能只因為是表格資料就判定需要資料庫服務。

</details>

## 正式資料與分析副本的分工

正式資料是業務目前採用並持續維護的紀錄；分析副本是為了計算報表而另外取得的資料。
例如資料庫保存訂單，匯出副本交給 pandas 計算，再由 matplotlib 畫出各月銷售。

修改分析副本不會自動更正正式資料。報表應記錄資料取得時間、篩選範圍及處理步驟，才知道它代表哪一批資料。
正式資料需要更正時，應走原系統的修改流程，不能把整理過的舊副本整份覆蓋回去。

問題：上午匯出的訂單檔，能直接代表下午的即時庫存嗎？

<details><summary>參考方向</summary>

不能，期間可能已有新的購買或取消。先確認報表需要多新的資料，再安排重新取得或更新副本的方式。
分析副本與快取都要處理來源更新，但分析副本不一定是為了加快重複讀取而建立。

</details>

## 記錄實際使用的環境

`sys.version` 是 Python 版本文字；本例三個套件以 `__version__` 提供版本。
`import matplotlib` 載入套件本身，讓這一格查詢套件版本。
`.split()` 依空白拆開文字，`[0]` 取得第一段版本號。

In [ ]:
import sys
import matplotlib
print("Python", sys.version.split()[0])
print("NumPy", np.__version__)
print("pandas", pd.__version__)
print("matplotlib", matplotlib.__version__)

## AI 程式審查所用的 Python 操作

`float(x)` 把數值或數字文字轉成浮點數；轉換不適用的型別會引發 TypeError，非法數字文字會引發 ValueError。
`except (TypeError, ValueError)` 捕捉這兩種轉換錯誤。
bool 雖能轉成 0 或 1，但本例不把它當成成績。
`sum(row["score"] for row in rows)` 使用生成器運算式，逐列提供 score 給 sum，不先建 list。
此處用 `except Exception` 顯示初稿的各種錯誤，讓所有測試都能觀察；不表示正式應用應忽略所有錯誤。

## AI 審查：先固定成績摘要的契約

情境：請 AI 撰寫成績摘要函數，先以已知答案的案例驗收，再檢查實作。

- 每列是一個 dict，score 可為數值、數字文字、None 或非法文字。
- 只接受能轉成數字、且落在 0–100 的值；bool、缺鍵及非法值略過。
- 回傳 `{"n": 有效筆數, "avg": 一位小數平均}`；沒有有效成績時 avg=None。
- 不修改輸入；同一案例重跑應有相同結果。

先執行初稿，指出正常、混合型別、空資料與界外值各發生什麼，再修改程式。

In [ ]:
# AI 初稿：先執行案例，記錄不符合契約的原因
def ai_draft_summary(rows):
    avg = sum(row["score"] for row in rows) / len(rows)
    return {"n": len(rows), "avg": round(avg, 1)}

draft_cases = [
    ("正常", [{"score": 80}, {"score": 90}], {"n": 2, "avg": 85.0}),
    ("混合型別", [{"score": "88"}, {"score": None}, {"score": 92}],
     {"n": 2, "avg": 90.0}),
    ("空資料", [], {"n": 0, "avg": None}),
    ("越界", [{"score": -1}, {"score": 100}, {"score": 101}],
     {"n": 1, "avg": 100.0}),
]

for label, rows_in, expected in draft_cases:
    try:
        got = ai_draft_summary(rows_in)
        check(got, expected, f"AI 初稿／{label}")
    except Exception as err:
        print(f"AI 初稿／{label}：❌ {type(err).__name__}: {err}")



In [ ]:
# 修正成績摘要函數
def summarize_scores(rows):
    """回傳符合成績規則的筆數與平均。"""
    ...   # TODO

acceptance_cases = [
    ("正常", [{"score": 80}, {"score": 90}], {"n": 2, "avg": 85.0}),
    ("混合型別", [{"score": "88"}, {"score": None}, {"score": 92}, {"score": "bad"}],
     {"n": 2, "avg": 90.0}),
    ("空資料", [], {"n": 0, "avg": None}),
    ("邊界", [{"score": -1}, {"score": 0}, {"score": 100}, {"score": 101}],
     {"n": 2, "avg": 50.0}),
]

for label, rows_in, expected in acceptance_cases:
    try:
        before_rows = [dict(row) for row in rows_in]
        check(summarize_scores(rows_in), expected, f"修正版／{label}")
        check(rows_in, before_rows, f"不修改輸入／{label}")
    except Exception as err:
        print(f"修正版／{label}：❌ 不應拋出 {type(err).__name__}: {err}")

<details><summary>AI 修正參考解答</summary>

```python
def summarize_scores(rows):
    valid = []
    for row in rows:
        raw = row.get("score")
        if type(raw) is bool:
            continue
        try:
            score = float(raw)
        except (TypeError, ValueError):
            continue
        if 0 <= score <= 100:
            valid.append(score)
    avg = round(sum(valid) / len(valid), 1) if valid else None
    return {"n": len(valid), "avg": avg}
```

增加缺鍵、bool 與小數字串案例，核對計算結果及輸入未改變。

</details>

# 綜合練習

以 Python 自測檢查基本操作，再組合檔案讀寫、資料清理、統計與圖表，完成資料處理流程。

## Python 自測（8 題）

第八題先填入兩次呼叫當下的輸出預測，再執行核對。

In [ ]:
# 第 1 題：comprehension ＋ 排序
scores = {"S001": 88, "S002": 55, "S003": 74, "S004": 92, "S005": 59}

passing = []   # TODO：取出分數 >= 60 的學號，依分數由高到低排列

check(passing, ["S004", "S001", "S003"], "第 1 題")

In [ ]:
# 第 2 題：函數
def top_k(pairs, k):
    """pairs 形如 [("甲", 70), ("乙", 90), ...]；回傳分數最高的 k 筆，由高到低"""
    ...   # TODO

check(top_k([("甲", 70), ("乙", 90), ("丙", 80), ("丁", 60)], 2), [("乙", 90), ("丙", 80)], "第 2 題")

In [ ]:
# 第 3 題：dict 聚合（按系所把分數加總）
rows = [("統計", 88), ("資訊", 75), ("統計", 92), ("數學", 80), ("統計", 70), ("資訊", 95)]

dept_total = {}   # TODO：算出 {"統計": 250, "資訊": 170, "數學": 80}

check(dept_total, {"統計": 250, "資訊": 170, "數學": 80}, "第 3 題")

In [ ]:
# 第 4 題：讀 CSV（用內建 csv 模組，不用 pandas）
import csv, io

raw_csv = io.StringIO("sid,grade\nS001,88\nS002,55\nS003,74\n")

records = []   # TODO：解析成 [{"sid": "S001", "grade": 88}, ...]，grade 要是 int

check(records, [{"sid": "S001", "grade": 88}, {"sid": "S002", "grade": 55}, {"sid": "S003", "grade": 74}], "第 4 題")

In [ ]:
# 第 5 題：字串格式化
def grade_label(name, score):
    """回傳形如「林佳蓉：92 分（優等）」的字串；90 分以上優等、80 ≤ score < 90 甲等、其餘乙等"""
    ...   # TODO（提示：f-string ＋ if/elif）

check((grade_label("林佳蓉", 92), grade_label("陳威廷", 85), grade_label("吳孟軒", 70)),
      ("林佳蓉：92 分（優等）", "陳威廷：85 分（甲等）", "吳孟軒：70 分（乙等）"), "第 5 題")

In [ ]:
# 第 6 題：class
class GradeBook:
    def __init__(self):
        self.data = {}          # 學號 -> 分數 list

    def add(self, sid, score):
        ...   # TODO：把 score 加進該學號的清單

    def average(self, sid):
        ...   # TODO：回傳該學號平均；沒有資料回傳 None

gb = GradeBook()
gb.add("S001", 80)
gb.add("S001", 90)
gb.add("S002", 70)
check((gb.average("S001"), gb.average("S002"), gb.average("S999")), (85.0, 70.0, None), "第 6 題")

In [ ]:
# 第 7 題：例外處理
def safe_int(s):
    """把字串轉 int；轉不了就回傳 None"""
    ...   # TODO（提示：try / except ValueError）

check((safe_int("88"), safe_int("八十八"), safe_int(""), safe_int("-5")),
      (88, None, None, -5), "第 7 題")

In [ ]:
# 第 8 題：讀碼題——先填兩次呼叫當下的輸出預測，再執行
predicted_first = None   # TODO
predicted_second = None  # TODO
def enroll(name, roster=[]):
    roster.append(name)
    return roster

first_result = list(enroll("A"))
print(first_result)
second_result = list(enroll("B"))
print(second_result)
check((predicted_first, predicted_second), (first_result, second_result), "第 8 題")


# list(...) 保留每次呼叫完成當下的內容；請另外說明如何使兩次登記互不影響。

<details><summary>📖 自測參考解答</summary>

```python
# 1
passing = [sid for sid, s in sorted(scores.items(), key=lambda kv: -kv[1]) if s >= 60]
# 2
def top_k(pairs, k):
    return sorted(pairs, key=lambda p: -p[1])[:k]
# 3
dept_total = {}
for dept, s in rows:
    dept_total[dept] = dept_total.get(dept, 0) + s
# 4
records = [{"sid": r["sid"], "grade": int(r["grade"])} for r in csv.DictReader(raw_csv)]
# 5
def grade_label(name, score):
    if score >= 90:
        label = "優等"
    elif score >= 80:
        label = "甲等"
    else:
        label = "乙等"
    return f"{name}：{score} 分（{label}）"
# 6
class GradeBook:
    def __init__(self):
        self.data = {}
    def add(self, sid, score):
        self.data.setdefault(sid, []).append(score)
    def average(self, sid):
        xs = self.data.get(sid)
        return sum(xs) / len(xs) if xs else None
# 7
def safe_int(s):
    try:
        return int(s)
    except ValueError:
        return None
# 8：兩次呼叫當下的輸出
predicted_first = ["A"]
predicted_second = ["A", "B"]

def enroll_independent(name, roster=None):
    if roster is None:
        roster = []
    roster.append(name)
    return roster
```
</details>

### 依自測結果補強（[Practical Python Programming](https://github.com/dabeaz-course/practical-python)）

| 卡在哪 | 去讀 |
|---|---|
| 第 1、3 題（資料結構、comprehension） | [資料結構](https://github.com/dabeaz-course/practical-python/blob/master/Notes/02_Working_with_data/01_Datatypes.md)、[comprehension](https://github.com/dabeaz-course/practical-python/blob/master/Notes/02_Working_with_data/06_List_comprehension.md) |
| 第 2 題（函數、lambda、排序） | [函數參數與回傳值](https://github.com/dabeaz-course/practical-python/blob/master/Notes/03_Program_organization/02_More_functions.md)、[lambda 與排序（PP §7.2）](https://github.com/dabeaz-course/practical-python/blob/master/Notes/07_Advanced_Topics/02_Anonymous_function.md) |
| 第 4 題（檔案與 CSV） | [檔案讀寫](https://github.com/dabeaz-course/practical-python/blob/master/Notes/01_Introduction/06_Files.md)、[csv.DictReader](https://docs.python.org/3/library/csv.html#csv.DictReader) |
| 第 5 題（字串、f-string） | [字串](https://github.com/dabeaz-course/practical-python/blob/master/Notes/01_Introduction/04_Strings.md)、[格式化輸出](https://github.com/dabeaz-course/practical-python/blob/master/Notes/02_Working_with_data/03_Formatting.md) |
| 第 6 題（class） | [class 與物件](https://github.com/dabeaz-course/practical-python/blob/master/Notes/04_Classes_objects/01_Class.md) |
| 第 7 題（例外） | [例外處理（PP §3.3）](https://github.com/dabeaz-course/practical-python/blob/master/Notes/03_Program_organization/03_Error_checking.md) |
| 第 8 題（可變預設參數） | [預設參數與可變物件](https://docs.python.org/3/tutorial/controlflow.html#default-argument-values) |

進一步練習：[Advanced Python Mastery](https://github.com/dabeaz-course/python-mastery)。

## 綜合練習：拼接課程資料

`course_grades` 有課號與成績，`course_lookup` 有課名與學分。請用一次 merge 算出每筆的
`quality_points = grade * credits`，最後把結果整理成
`[("資料庫管理", 264), ("統計學（一）", 380)]`。

In [ ]:
course_grades = pd.DataFrame({"cid": ["C103", "C101"], "grade": [88, 95]})
course_lookup = pd.DataFrame({
    "cid": ["C101", "C103"],
    "title": ["統計學（一）", "資料庫管理"],
    "credits": [4, 3],
})

merge_answer = []   # TODO
check(merge_answer, [("資料庫管理", 264), ("統計學（一）", 380)], "pandas 拼表")

<details><summary>參考解答</summary>

```python
detail = course_grades.merge(
    course_lookup, on="cid", how="left", validate="many_to_one")
detail["quality_points"] = detail["grade"] * detail["credits"]
merge_answer = list(detail[["title", "quality_points"]].itertuples(index=False, name=None))
```

驗收時再問兩句：左表若有不存在的 `cid`，結果會在哪裡出現 `NaN`？右表課號重複時，
`validate="many_to_one"` 為什麼應該阻止你繼續算？
</details>

## 綜合練習：建立可驗證的系所報表

完成 `build_data_report(score_rows, lookup)`。先驗證學號，再拼接系所；配對後系所缺值時引發 ValueError。
按系所計算 n、nonmissing_n、avg_grade，平均捨入至小數第二位，依 dept 升冪回傳這四欄。
函數不可修改輸入。缺值仍算一筆紀錄，但不納入平均。使用課內的 report_scores 與 student_lookup 測試。

將這份報表寫到 data_dir 的 CSV，再畫各系非缺值成績數；圖中數值取自自己的結果。

In [ ]:
def build_data_report(score_rows, lookup):
    ...  # TODO：驗證、拼接、分組、排序，回傳 DataFrame

report_answer = build_data_report(report_scores, student_lookup)
if report_answer is None:
    print("待完成：data_report")
else:
    report_records = list(report_answer[["dept", "n", "nonmissing_n", "avg_grade"]].itertuples(index=False, name=None))
    check(report_records, [("統計", 4, 3, 78.0), ("資訊", 1, 1, 74.0)], "data_report")
    # TODO：將 report_answer 寫成 CSV，再畫各系非缺值成績數。

<details><summary>參考解答</summary>

```python
def build_data_report(score_rows, lookup):
    validate_student_keys(score_rows, lookup)
    detail = score_rows.merge(lookup, on="sid", how="left", validate="many_to_one")
    if detail["dept"].isna().any():
        raise ValueError("部分成績缺少系所資料")
    report = detail.groupby("dept", as_index=False).agg(
        n=("grade", "size"), nonmissing_n=("grade", "count"), avg_grade=("grade", "mean"))
    report["avg_grade"] = report["avg_grade"].round(2)
    return report.sort_values("dept")

report_answer = build_data_report(report_scores, student_lookup)
report_records = list(report_answer[["dept", "n", "nonmissing_n", "avg_grade"]].itertuples(index=False, name=None))
check(report_records, [("統計", 4, 3, 78.0), ("資訊", 1, 1, 74.0)], "data_report")
report_answer.to_csv(data_dir / "department_report.csv", index=False, encoding="utf-8")
report_labels = report_answer["dept"].map({"統計": "Statistics", "資訊": "CS"})
assert report_labels.notna().all()
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(report_labels, report_answer["nonmissing_n"])
axes[0].set_title("Non-missing grade count")
axes[1].bar(report_labels, report_answer["avg_grade"])
axes[1].set_title("Mean of non-missing grades")
plt.tight_layout()
fig.savefig(data_dir / "department_report.png")
plt.show()
plt.close(fig)
```
</details>

## 自己的資料處理流程

選擇收藏、器材或其他熟悉主題，用 CSV 保存至少四個欄位與五筆資料。
說明一列代表什麼，將讀取、驗證與統計分成函數，保留無效及重複資料的處理結果。

重讀輸出檔，核對完整內容；用 pandas 完成一項分組報告，以 matplotlib 畫出結果。
至少測試正常資料、空輸入、非法值及重複識別碼。

說明哪些規則由驗證函數檢查；若其他程式直接改寫 CSV，這些規則是否仍會執行？
依照課內的兩份副本案例，指出兩人同時編輯時，僅通過資料驗證仍不足以保護哪一項修改。

### 進階練習（選做）



| 主題 | 任務 | 這一步多練什麼 |
|---|---|---|
| f-string | 把金額格式化成 `NT$ 1,234,567.50` | 千分位＋兩位小數 |
| list 與切片 | 用切片去掉首尾觀測 | 邊界與「含頭不含尾」 |
| dict、tuple、set | 找出已登記但尚未完成者 | set 差集 |
| 流程控制 | 一次走訪，把學號分進及格／不及格 | 分支＋累積容器 |
| comprehension | 調分後夾在 0–100，建成學號→分數 dict | 轉換、dict comprehension |
| 函數 | 寫可檢查輸入的加權平均 | 函數契約與 guard clause |
| 例外處理 | 接受整數或字串，拒絕空值、亂碼與越界成績 | 型別檢查＋邊界 |



加權平均遇到空輸入、長度不符或權重總和為零時回傳 None。
`safe_grade` 只接受 int 或整數文字，不接受 bool、小數或 None；`type(raw) is int` 可排除 bool。
`int(True)` 會得到 1，`int(88.9)` 會得到 88；成功轉換不等於符合輸入型別規則。
`or` 表示任一條件成立，`and` 要求全部成立；`tuple(清單)` 依清單內容建立 tuple。

In [ ]:
# 進階 1：f-string
amount = 1234567.5
amount_text = ""                         # TODO
check(amount_text, "NT$ 1,234,567.50", "進階 1")

In [ ]:
# 進階 2：list 與切片
observations = [45, 72, 88, 91, 60, 77]
inner = []                               # TODO：只用一次切片
check(inner, [72, 88, 91, 60], "進階 2")

In [ ]:
# 進階 3：dict、tuple、set
registered = {"S001", "S002", "S003", "S004"}
completed = {"S001", "S003"}
pending = set()                          # TODO
check(pending, {"S002", "S004"}, "進階 3")

In [ ]:
# 進階 4：流程控制
adv_rows = [("S001", 88), ("S002", 55), ("S003", 74)]
groups = {"pass": [], "fail": []}        # TODO：用一個 for 填入學號
check(groups, {"pass": ["S001", "S003"], "fail": ["S002"]}, "進階 4")

In [ ]:
# 進階 5：comprehension
raw_scores = {"S001": 96, "S002": 52, "S003": -8}
adjusted = {}                            # TODO：每筆 +5，再夾在 0–100
check(adjusted, {"S001": 100, "S002": 57, "S003": 0}, "進階 5")

In [ ]:
# 進階 6：函數
def weighted_mean(values, weights):
    """長度不同、空輸入或權重總和為零時回 None，否則回加權平均。"""
    ...                                  # TODO

check((weighted_mean([80, 90], [1, 3]), weighted_mean([], []),
       weighted_mean([80], [1, 2])), (87.5, None, None), "進階 6")

In [ ]:
# 進階 7：例外處理
def safe_grade(raw):
    """整數或整數字串且落在 0–100 才回 int，其他一律回 None。"""
    ...                                  # TODO

check(tuple([safe_grade(x) for x in (88, "72", None, "七十", 101, 88.9, True)]),
      (88, 72, None, None, None, None, None), "進階 7")

<details><summary>📖 進階練習參考解答</summary>

```python
# 進階 1
amount_text = f"NT$ {amount:,.2f}"
# 進階 2
inner = observations[1:-1]
# 進階 3
pending = registered - completed

# 進階 4
groups = {"pass": [], "fail": []}
for sid, score in adv_rows:
    groups["pass" if score >= 60 else "fail"].append(sid)

# 進階 5
adjusted = {sid: min(100, max(0, score + 5))
            for sid, score in raw_scores.items()}

# 進階 6
def weighted_mean(values, weights):
    if not values or len(values) != len(weights) or sum(weights) == 0:
        return None
    return sum([x * w for x, w in zip(values, weights)]) / sum(weights)

# 進階 7
def safe_grade(raw):
    if not (type(raw) is int or type(raw) is str):
        return None
    try:
        score = int(raw)
    except ValueError:
        return None
    return score if 0 <= score <= 100 else None
```

</details>

## 查閱資料

- [Practical Python Programming](https://github.com/dabeaz-course/practical-python)：依自測結果補練 Python。
- [Python 標準函式庫](https://docs.python.org/3/library/)：csv、json、例外與檔案操作。
- [NumPy](https://numpy.org/doc/stable/user/absolute_beginners.html)：陣列與基本運算。
- [pandas](https://pandas.pydata.org/docs/user_guide/10min.html)：表格操作。
- [matplotlib](https://matplotlib.org/stable/users/explain/quick_start.html)：繪圖介面。
- [專題規範](https://github.com/chang-ye-tu/db/blob/master/projects.md)：自己的題目需求與驗收要求。